# 07 — Correção das variáveis de histórico do vendedor

Nos notebooks anteriores, duas variáveis do vendedor tinham problemas:

- **`taxa_atraso_vendedor`**: contava como "atrasados" pedidos anteriores que, na data da compra, **ainda não tinham sido entregues**. Ou seja, usava uma informação que não existia no momento da compra (vazamento sutil).
- **`volume_historico_vendedor`**: só cresce com o tempo (é a soma de todos os pedidos anteriores), então funciona como um "relógio". Como o teste está no futuro, as árvores veem valores maiores do que qualquer um do treino e não conseguem extrapolar.

Neste notebook:

1. Recalculamos a taxa de atraso usando **só os pedidos que já tinham sido entregues até o momento da compra**.
2. Trocamos o volume histórico pelo **número de pedidos do vendedor nos 30 dias anteriores à compra**.
3. Rodamos de novo a Regressão Linear (baseline), a Random Forest e o XGBoost otimizados, com a mesma divisão temporal, e comparamos **antes × depois**.
4. Refazemos a análise de importância das variáveis do XGBoost.

## I. Importações e configurações

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from xgboost import XGBRegressor

def root_mean_squared_error(y_real, y_prev):
    return np.sqrt(mean_squared_error(y_real, y_prev))

# Pastas: ajuste aqui se a estrutura do repositório for diferente
PASTAS_RAW = ["../raw", "../raw/olist_raw", "../data/raw", "../datasets"]
CAMINHO_MODELO = "../data/processed/df_modelo.csv"
CAMINHO_SAIDA = "../data/processed/df_modelo_v2.csv"

DATA_CORTE = "2018-04-01"   # mesma divisão temporal dos notebooks 05 e 06
JANELA_DIAS = 30

def acha_pasta_raw():
    raiz = Path.cwd().parent       
    achados = list(raiz.rglob("olist_orders_dataset.csv"))
    if not achados:
        print("Pasta atual:", Path.cwd())
        print("Conteúdo de raw:", list((raiz / "raw").glob("*")) if (raiz / "raw").exists() else "pasta raw não existe")
        raise FileNotFoundError("olist_orders_dataset.csv não está em nenhuma pasta do projeto. Extraia o olist_raw.zip dentro de raw/.")
    return str(achados[0].parent)
PASTA_RAW = acha_pasta_raw()
print("Dados brutos em:", PASTA_RAW)

## II. Leitura dos dados brutos

Precisamos das datas originais (compra, entrega e prazo estimado) de **todos** os pedidos, e do vendedor de cada item.

In [ ]:
pedidos = pd.read_csv(
    os.path.join(PASTA_RAW, "olist_orders_dataset.csv"),
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                 "order_estimated_delivery_date"],
)
itens = pd.read_csv(os.path.join(PASTA_RAW, "olist_order_items_dataset.csv"))

print("Pedidos:", pedidos.shape, "| Itens:", itens.shape)

## III. Vendedor principal de cada pedido

Seguimos o mesmo critério do notebook de engenharia de atributos: o vendedor principal é o do item mais caro do pedido.

In [ ]:
vendedor_principal = (
    itens.sort_values(["order_id", "price"], ascending=[True, False])
         .drop_duplicates("order_id")[["order_id", "seller_id"]]
)

# Todos os pares (pedido, vendedor): um pedido com dois vendedores conta no histórico dos dois
pares = itens[["order_id", "seller_id"]].drop_duplicates()
pares = pares.merge(
    pedidos[["order_id", "order_purchase_timestamp", "order_delivered_customer_date",
             "order_estimated_delivery_date"]],
    on="order_id", how="left",
)
pares["atrasou"] = (
    pares["order_delivered_customer_date"].dt.normalize() > pares["order_estimated_delivery_date"]
).astype(int)

print("Pedidos com vendedor principal:", len(vendedor_principal))

## IV. Nova taxa de atraso do vendedor

Para cada pedido, olhamos o vendedor e contamos apenas os pedidos dele que **já tinham chegado ao cliente antes da data da compra**. A taxa é:

taxa de atraso = (pedidos entregues com atraso até a compra) / (pedidos entregues até a compra)

Usamos merge_asof, que para cada compra procura a última entrega do mesmo vendedor que aconteceu antes dela, já com os totais acumulados até ali. Assim, um pedido ainda em trânsito nunca entra na conta.

Vendedores sem nenhuma entrega anterior ficam sem taxa; marcamos esses casos com vendedor_sem_historico=1 e preenchemos a taxa com a média do treino.

In [ ]:
# Linha do tempo de entregas de cada vendedor, com contagens acumuladas
entregas = (
    pares.dropna(subset=["order_delivered_customer_date"])
         .sort_values("order_delivered_customer_date")
         .copy()
)
entregas["entregues_acum"] = entregas.groupby("seller_id").cumcount() + 1
entregas["atrasados_acum"] = entregas.groupby("seller_id")["atrasou"].cumsum()
entregas = entregas[["seller_id", "order_delivered_customer_date", "entregues_acum", "atrasados_acum"]]

# Momento da compra de cada pedido (vendedor principal)
compras = vendedor_principal.merge(
    pedidos[["order_id", "order_purchase_timestamp"]], on="order_id"
).sort_values("order_purchase_timestamp")

# Última entrega ESTRITAMENTE anterior à compra, do mesmo vendedor
hist = pd.merge_asof(
    compras, entregas,
    left_on="order_purchase_timestamp", right_on="order_delivered_customer_date",
    by="seller_id", direction="backward", allow_exact_matches=False,
)
hist["entregues_acum"] = hist["entregues_acum"].fillna(0)
hist["atrasados_acum"] = hist["atrasados_acum"].fillna(0)

hist["vendedor_sem_historico"] = (hist["entregues_acum"] == 0).astype(int)
hist["taxa_atraso_vendedor_corr"] = np.where(
    hist["entregues_acum"] > 0, hist["atrasados_acum"] / hist["entregues_acum"], np.nan
)
hist["entregas_anteriores_vendedor"] = hist["entregues_acum"]

hist[["taxa_atraso_vendedor_corr", "entregas_anteriores_vendedor", "vendedor_sem_historico"]].describe()